# Minimal working example using BERT (`distBERT` model)

In [ ]:
!pip install pandas torch transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.4 MB/s eta 0:00:00


## Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [ ]:
# ----------------------------
# 1) Sample pseudo "data"
# ----------------------------
#items = pd.DataFrame([
#    {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
#    {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
#    {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
#    {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
#    {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
#])
#
#events = pd.DataFrame([
#    {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
#    {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
#    {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
#    {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
#    {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
#])
#events["ts"] = pd.to_datetime(events["ts"])

items = pd.read_csv("items.csv")
events = pd.read_csv ("events.csv")
events["ts"] = pd.to_datetime(events["ts"])


In [ ]:
items.head()

,item_id,title,description,category,brand,price,tags,text
0,i1,Noise Cancelling Headphones,Wireless noise-cancelling headphones with 30-h...,electronics,SoundPeak,199.99,"audio,wireless,travel,premium",Noise Cancelling Headphones. Wireless noise-ca...
1,i2,Mechanical Keyboard,"Mechanical keyboard with RGB backlighting, hot...",electronics,KeyForge,129.00,"keyboard,gaming,productivity,desk",Mechanical Keyboard. Mechanical keyboard with ...
2,i3,Running Shoes,Running shoes designed for long distance comfo...,sports,StrideLab,110.00,"running,fitness,outdoors,comfort",Running Shoes. Running shoes designed for long...
3,i4,Vegetarian Cookbook,"Cookbook featuring quick vegetarian recipes, p...",books,HomeTable Press,24.99,"cooking,vegetarian,recipes,home",Vegetarian Cookbook. Cookbook featuring quick ...
4,i5,Fitness Smartwatch,"Smartwatch with heart rate monitoring, GPS, sl...",electronics,PulsePath,249.00,"wearables,fitness,gps,health",Fitness Smartwatch. Smartwatch with heart rate...


In [ ]:
events.head()

,user_id,item_id,ts,event_type,session_id,device,dwell_seconds,price_at_event,category_at_event
0,u1,i2,2026-01-01 07:47:00,click,s_u1_20260101_1,tablet,56,129.00,electronics
1,u1,i13,2026-01-04 08:01:00,view,s_u1_20260104_2,tablet,7,149.99,electronics
2,u1,i2,2026-01-07 14:28:00,click,s_u1_20260107_3,mobile,72,129.00,electronics
3,u1,i13,2026-01-07 19:06:00,view,s_u1_20260107_1,tablet,67,149.99,electronics
4,u1,i1,2026-01-10 08:42:00,add_to_cart,s_u1_20260110_1,desktop,46,199.99,electronics


## BERT encoder (What makes BERT work?)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [ ]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------
# first step is going from text to tokens
# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [ ]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = items["title"] + ". " + items["description"]  + ". "+ items["category"] + "."+ items["tags"] + "." + items["price"].astype(str)
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)

## What user-specific data are there?

In [ ]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
weights_mapping = { "click" : 3,"view": 1  ,"add_to_cart" : 5, "purchase" : 10}

events['event_type_enc'] = events['event_type'].map(weights_mapping)

def build_user_text(user_id, events, items, N=3):
    hist = (events[events["user_id"] == user_id]
            .sort_values(["ts"])
            .tail(N)
            .to_dict("records"))

    if not hist:
        return "no history", set()

    item_lookup = items.set_index("item_id")["text"]
    texts = []
    seen = set()

    for row in hist:
        if row["item_id"] in item_lookup.index:
            seen.add(row["item_id"])
            texts.extend([item_lookup[row["item_id"]]] * row["event_type_enc"])

    return " ".join(texts), seen


## How to recommend "similar" item with BERT?

In [ ]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [ ]:

for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i6', 'i13', 'i14']
u2 -> ['i18', 'i13', 'i11']
u3 -> ['i8', 'i4', 'i6']
